## Imports

In [1]:
from qsopt import * 
import numpy as np
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt

## Define experimental parameters

In [2]:
# Define custom physical constants
custom_constants = PhysicalConstants(
    chi=0.75,                    # Dispersive coupling
    photon_cavity_coupling=1.5,  # Photon-cavity coupling
    inverse_pulse_width=0.2      # Inverse pulse width
)

# Define custom system dimensions
custom_dims = SystemDimensions(
    cavity_levels=2,
    qubit_levels=2,
    field_levels=2
)

# Define measurement protocol
custom_measurement = MeasurementProtocol(
    measurement_times = [-5.0, 0.0, 5.0]
)

# Define initial state configuration (SINGLE_PHOTON)
initial_state = InitialStateConfig(
    state_type=InitialStateType.SINGLE_PHOTON
)

# Define noise configuration
noise_config = NoiseConfiguration(
    depolarizing=0.01,  
    dephasing=0.005,      
    relaxation=0.01     
)

# Create parameters with custom configuration
exp_parameters = ExperimentalParameters(
    physical_constants=custom_constants,
    system_dims=custom_dims,
    measurement=custom_measurement,
    initial_state=initial_state,
    noise_config=noise_config
)

print(exp_parameters)

SYSTEM DIMENSIONS
  Cavity levels:             2
  Qubit levels:              2
  Field levels:              2
  Total dimension:           8
  Status:               VALID
PHYSICAL CONSTANTS
  Chi:                    0.7500
  Photon cavity coupling: 1.5000
  Inverse pulse width:    0.2000
  Status:               VALID
MEASUREMENT PROTOCOL
  Number of measurements:      3
  Measurement times: [-5.0, 0.0, 5.0]
  Status:               VALID
INITIAL STATE
  Type:                 single_photon
NOISE MODEL
  Depolarizing rate:      0.0100
  Dephasing rate:         0.0050
  Relaxation rate:        0.0100
  Custom operators:     None
  Status:               VALID
SYSTEM STATUS
  Configuration:        VALID


## Define trainable parameters

In [3]:
parameters = TrainableParameters()
parameters.add_rotation_angles(['ry1', 'ry2'], [1., 1.2], optimizer=optax.adam(0.01))

print(parameters)

TrainableParameters(total=2)
  Rotation Angles:
    ry1: 1.0000 rad (57.30°)
    ry2: 1.2000 rad (68.75°)


## Define experiment

In [4]:
experiment = SingleQubitExperiment(exp_parameters, parameters)

## Test Single Simulation

In [5]:
results = experiment.run_simulation()

print(f"\nCurrent parameters:")
print(f"{results['ry1']:.4f} rad, {results['ry2']:.4f} rad")

print(f"\nSimulation results:")
print(f"  P(with interaction):    {results['prob_with']:.6f}")
print(f"  P(without interaction): {results['prob_without']:.6f}")
print(f"  Contrast:               {results['contrast']:.6f}")

# Store initial contrast for comparison later
initial_contrast = results['contrast']


Current parameters:
1.0000 rad, 1.2000 rad

Simulation results:
  P(with interaction):    0.891351
  P(without interaction): 0.771814
  Contrast:               0.119537


## Run Optimization

In [ ]:
# Run optimization for 20 steps
# The experiment automatically uses its built-in callback (saves every epoch by default)
print("Starting optimization...\n")

callback = experiment.optimize(
    num_steps=20,
    learning_rate=0.05,
    verbose=True
)

print("\nOptimization complete!")
print(f"Callback recorded {len(callback.history['epochs'])} epochs")
print(f"Converged: {callback.converged}")
print(f"Final gradient norm: {callback.final_grad_norm:.2e}")

Starting optimization...

Starting optimization...
Initial: θ₁=1.000 rad, θ₂=1.000 rad
Step  θ₁          θ₂          Contrast    Loss        Grad Norm
----------------------------------------------------------------------


TypeError: unsupported operand type(s) for *: 'Qobj' and 'JVPTracer'

## Visualize Optimization Progress

Use the callback data to plot the optimization trajectory.

In [ ]:
# Access the callback (returned from optimize)
# Note: callback is the same as experiment.callback

# Create figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Contrast vs Epoch
ax = axes[0]
ax.plot(callback.history['epochs'], callback.history['contrast'], 'b-', linewidth=2, label='Contrast')
ax.axhline(y=callback.best_metrics['contrast'], color='r', linestyle='--', label='Best contrast')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Contrast', fontsize=12)
ax.set_title('Sensing Contrast Evolution', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Detection Probabilities
ax = axes[1]
ax.plot(callback.history['epochs'], callback.history['prob_with'], 'g-', linewidth=2, label='P(with photon)')
ax.plot(callback.history['epochs'], callback.history['prob_without'], 'orange', linewidth=2, label='P(without photon)')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Detection Probabilities', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Parameters trajectory
ax = axes[2]
# Extract parameter arrays from trainable_params history
param_arrays = []
for tp in callback.history['trainable_params']:
    angles = tp.get_rotation_angles()
    param_arrays.append([angles[name][0] for name in angles.keys()])
params_array = np.array(param_arrays)

ax.plot(callback.history['epochs'], params_array[:, 0], 'b-', linewidth=2, label='ry1')
ax.plot(callback.history['epochs'], params_array[:, 1], 'r-', linewidth=2, label='ry2')

# Get best parameters
best_tp = callback.get_best_trainable_params()
best_angles = best_tp.get_rotation_angles()
best_params = [best_angles[name][0] for name in best_angles.keys()]
ax.axhline(y=best_params[0], color='b', linestyle='--', alpha=0.5)
ax.axhline(y=best_params[1], color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Parameter Value (rad)', fontsize=12)
ax.set_title('Parameter Trajectories', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nOptimization Summary:")
print(f"  Total iterations: {callback.epoch}")
print(f"  Converged: {callback.converged}")
print(f"  Final gradient norm: {callback.final_grad_norm:.2e}")
print(f"  Best epoch: {callback.best_metrics['epoch']}")
print(f"  Best contrast: {callback.best_metrics['contrast']:.6f}")
print(f"  Best parameters: ry1={best_params[0]:.4f} rad, ry2={best_params[1]:.4f} rad")

## Save and Load Callback Data

Save the optimization results for future analysis and plotting.

## Custom Callback (Optional)

You can also provide a custom callback to control save frequency or other behavior.

In [ ]:
# Example: Create a custom callback that saves every 5 epochs
custom_callback = OptimizationCallback(save_every=5, save_best=True)

# Run optimization with custom callback
callback2 = experiment.optimize(
    num_steps=15,
    learning_rate=0.03,
    verbose=True,
    callback=custom_callback  # Use custom callback
)

print(f"\nCustom callback info:")
print(f"  Total epochs: {callback2.epoch}")
print(f"  Saved epochs: {len(callback2.history['epochs'])} (saved every 5th epoch)")
print(f"  Converged: {callback2.converged}")
print(f"  Best contrast: {callback2.best_contrast:.6f}")

# Note: callback2 is the same instance as custom_callback
assert callback2 is custom_callback

In [ ]:
# Save callback results
callback.save('optimization_results.npz')
print("Saved optimization results to 'optimization_results.npz'")

# Load the results (demonstrating how to reload for future use)
loaded_data = OptimizationCallback.load('optimization_results.npz')

print("\nLoaded data contains:")
for key in loaded_data.keys():
    if hasattr(loaded_data[key], 'shape'):
        print(f"  {key}: shape {loaded_data[key].shape}")
    else:
        print(f"  {key}: {loaded_data[key]}")

# Example: Plot contrast from loaded data
plt.figure(figsize=(8, 5))
plt.plot(loaded_data['epochs'], loaded_data['contrast'], 'b-', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Contrast')
plt.title('Sensing Contrast (from loaded data)')
plt.grid(True, alpha=0.3)
plt.show()

## View Results

In [ ]:
# Get final values directly from trainable_params (they're updated during optimization)
final_results = experiment.run_simulation()

print("="*60)
print("OPTIMIZATION SUMMARY")
print("="*60)
print(f"\nInitial Parameters:")
print(f"  θ₁ = {np.pi/2:.4f} rad = {np.degrees(np.pi/2):.2f}°")
print(f"  θ₂ = {-np.pi/2:.4f} rad = {np.degrees(-np.pi/2):.2f}°")
print(f"  Contrast = {initial_contrast:.6f}")

print(f"\nFinal Parameters:")
print(f"  θ₁ = {final_results['ry1']:.4f} rad = {np.degrees(final_results['ry1']):.2f}°")
print(f"  θ₂ = {final_results['ry2']:.4f} rad = {np.degrees(final_results['ry2']):.2f}°")
print(f"  Contrast = {final_results['contrast']:.6f}")

if abs(initial_contrast) > 1e-10:
    improvement = ((final_results['contrast'] - initial_contrast) / abs(initial_contrast)) * 100
    print(f"\nImprovement: {improvement:+.2f}%")
else:
    print(f"\nImprovement: N/A (initial contrast near zero)")
print("="*60)

## Manually Update Parameters

You can also manually update parameter values and re-run simulations:

In [ ]:
# Update parameters directly in the trainable_params object
parameters.parameters[0].value = 0.0  # Set θ₁ to 0
parameters.parameters[1].value = np.pi  # Set θ₂ to π

# Run simulation with new values
new_results = experiment.run_simulation()

print(f"Testing with θ₁ = {new_results['ry1']:.4f}, θ₂ = {new_results['ry2']:.4f}")
print(f"Contrast: {new_results['contrast']:.6f}")

# Restore optimized values
parameters.parameters[0].value = final_results['ry1']
parameters.parameters[1].value = final_results['ry2']
print(f"\nRestored optimized parameters")